# Lección 17 - Creación de Agentes de IA Locales con Foundry Local y Qwen

En este cuaderno construyes un **asistente de ingeniería local** que se ejecuta completamente en tu estación de trabajo. No se utiliza inferencia en la nube en ningún momento. El asistente puede:

1. **Llamar a herramientas** a través de llamadas a funciones Qwen mediante Foundry Local.
2. **Listar y leer archivos** dentro de un directorio de proyecto sandbox.
3. **Analizar código** con métricas simples.
4. **Buscar documentación** con RAG local (Chroma).
5. **Usar un servidor MCP local** (se omite de manera elegante si no está configurado).

El código del agente se ve casi idéntico a las lecciones en la nube — solo que el endpoint del cliente se mueve de la nube a `localhost`.


## Configuración

Antes de ejecutar este cuaderno:

1. **Instale Microsoft Foundry Local** (consulte la [documentación](https://learn.microsoft.com/azure/ai-foundry/foundry-local/) para su sistema operativo).
2. **Descargue y ejecute un modelo Qwen:**
   ```bash
   foundry model run qwen2.5-7b-instruct
   foundry service status
   ```
3. Instale los paquetes de Python a continuación.

Todo se ejecuta localmente. Una máquina con ~8 GB de RAM es un mínimo realista.


In [ ]:
%pip install foundry-local-sdk openai chromadb -q

## 1. Conectar a Foundry Local

`FoundryLocalManager` descarga el modelo si es necesario, inicia el servicio local y nos proporciona un **endpoint compatible con OpenAI**. Luego, apuntamos el SDK estándar de OpenAI hacia él. La clave API es un marcador local: no se involucra ninguna credencial en la nube.


In [ ]:
from foundry_local import FoundryLocalManager
from openai import OpenAI

MODEL_ALIAS = "qwen2.5-7b-instruct"

# Foundry Local selects the best build for your hardware (CPU / GPU / NPU) automatically.
manager = FoundryLocalManager(MODEL_ALIAS)
model_info = manager.get_model_info(MODEL_ALIAS)

client = OpenAI(
    base_url=manager.endpoint,   # e.g. http://localhost:PORT/v1
    api_key=manager.api_key,     # local placeholder
)

MODEL_ID = model_info.id
print(f"Connected to Foundry Local. Serving: {MODEL_ID}")
print(f"Endpoint: {manager.endpoint}")

In [ ]:
# Quick sanity check: a plain chat completion, running fully on-device.
resp = client.chat.completions.create(
    model=MODEL_ID,
    messages=[{"role": "user", "content": "In one sentence, what is a small language model?"}],
)
print(resp.choices[0].message.content)

## 2. Herramientas Locales (Operaciones de Archivos en un Entorno Aislado)

Creamos un pequeño proyecto de ejemplo en el disco, luego definimos herramientas limitadas a la raíz de ese proyecto. La verificación del entorno aislado es importante incluso localmente: una herramienta que lee rutas arbitrarias se ejecuta con los permisos de tu usuario y puede acceder a todo lo que tú puedes.


In [ ]:
import json
from pathlib import Path

# Create a tiny sample project so the notebook is self-contained.
PROJECT_ROOT = Path.cwd() / "sample_project"
PROJECT_ROOT.mkdir(exist_ok=True)

(PROJECT_ROOT / "auth.py").write_text(
    '"""Authentication helpers."""\n\n'
    "def login(user, password):\n"
    "    # TODO: hash the password before comparing\n"
    "    return user == 'admin' and password == 'secret'\n\n"
    "def logout(session):\n"
    "    session.clear()\n",
    encoding="utf-8",
)
(PROJECT_ROOT / "utils.py").write_text(
    '"""Utility functions."""\n\n'
    "def clamp(value, low, high):\n"
    "    return max(low, min(value, high))\n",
    encoding="utf-8",
)
print("Sample project created at:", PROJECT_ROOT)

In [ ]:
def _safe_path(path: str) -> Path | None:
    """Resolve a path and confirm it stays inside the project sandbox."""
    full = (PROJECT_ROOT / path).resolve()
    if full == PROJECT_ROOT or PROJECT_ROOT in full.parents:
        return full
    return None


def list_files() -> str:
    """List files in the project directory."""
    files = [p.name for p in PROJECT_ROOT.iterdir() if p.is_file()]
    return ", ".join(files) if files else "(no files)"


def read_file(path: str) -> str:
    """Read a file, but only inside the sandboxed project directory."""
    full = _safe_path(path)
    if full is None:
        return "Access denied: path is outside the project directory."
    if not full.is_file():
        return f"No such file: {path}"
    return full.read_text(encoding="utf-8")


def analyze_code(path: str) -> str:
    """Report simple metrics about a source file."""
    full = _safe_path(path)
    if full is None or not full.is_file():
        return "File not found or access denied."
    text = full.read_text(encoding="utf-8")
    lines = text.splitlines()
    return json.dumps({
        "path": path,
        "lines": len(lines),
        "functions": sum(1 for ln in lines if ln.strip().startswith("def ")),
        "todos": sum(1 for ln in lines if "TODO" in ln or "FIXME" in ln),
    })


print(list_files())

## 3. RAG local con Chroma

Insertamos un pequeño conjunto de fragmentos de documentación en una colección local de Chroma. Chroma se ejecuta en proceso y almacena vectores en disco — sin servidor, sin nube. La herramienta `search_docs` recupera los fragmentos más relevantes para una consulta.


In [ ]:
import chromadb

DOCS = {
    "auth": "The login() function checks credentials. It currently compares passwords in plain text, which should be hashed.",
    "sessions": "Sessions are cleared on logout via session.clear(). Sessions are stored in memory and lost on restart.",
    "utils": "clamp(value, low, high) constrains a number to a range. Used throughout the UI layer for bounds checking.",
    "style": "This project follows PEP 8. Functions use snake_case and modules include a docstring at the top.",
}

# Chroma ships with a local default embedding model, so embedding stays on-device.
chroma_client = chromadb.Client()
collection = chroma_client.get_or_create_collection("project_docs")
collection.upsert(
    ids=list(DOCS.keys()),
    documents=list(DOCS.values()),
)


def search_docs(query: str) -> str:
    """Search the local documentation index for relevant snippets."""
    results = collection.query(query_texts=[query], n_results=2)
    docs = results.get("documents", [[]])[0]
    return "\n".join(docs) if docs else "No relevant documentation found."


print(search_docs("how are passwords handled?"))

## 4. El Bucle de Llamado de Herramientas

Ahora registramos las herramientas con el modelo usando el esquema de herramientas de OpenAI y ejecutamos el bucle estándar de llamado de herramientas: el modelo solicita una herramienta, la ejecutamos localmente, alimentamos el resultado de vuelta y repetimos hasta que el modelo produce una respuesta final. La llamada a funciones confiable de Qwen es lo que hace que esto funcione en el dispositivo.


In [ ]:
TOOLS_SCHEMA = [
    {"type": "function", "function": {
        "name": "list_files", "description": "List files in the project directory.",
        "parameters": {"type": "object", "properties": {}},
    }},
    {"type": "function", "function": {
        "name": "read_file", "description": "Read a file inside the project directory.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string", "description": "File name, e.g. auth.py"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "analyze_code", "description": "Report line count, function count and TODO count for a file.",
        "parameters": {"type": "object", "properties": {
            "path": {"type": "string"}}, "required": ["path"]},
    }},
    {"type": "function", "function": {
        "name": "search_docs", "description": "Search local documentation for a query.",
        "parameters": {"type": "object", "properties": {
            "query": {"type": "string"}}, "required": ["query"]},
    }},
]

TOOL_IMPL = {
    "list_files": list_files,
    "read_file": read_file,
    "analyze_code": analyze_code,
    "search_docs": search_docs,
}

SYSTEM_PROMPT = (
    "You are a local engineering assistant. Use the provided tools to inspect the project "
    "and its documentation. Prefer calling a tool over guessing. Be concise."
)

In [ ]:
def run_agent(user_query: str, max_iterations: int = 5) -> str:
    """Standard tool-calling loop, running entirely against the local model."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_query},
    ]

    for _ in range(max_iterations):
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            tools=TOOLS_SCHEMA,
        )
        msg = response.choices[0].message

        if not msg.tool_calls:
            return msg.content or "(no answer)"

        # Record the assistant's tool-call request.
        messages.append({
            "role": "assistant",
            "content": msg.content,
            "tool_calls": [tc.model_dump() for tc in msg.tool_calls],
        })

        # Execute each requested tool locally and feed results back.
        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments or "{}")
            result = TOOL_IMPL[name](**args) if name in TOOL_IMPL else f"Unknown tool: {name}"
            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": str(result),
            })

    return "Stopped: reached max tool-calling iterations."


print("Agent ready.")

In [ ]:
# A file-reading question.
print(run_agent("What does auth.py do, and is there anything to fix in it?"))

In [ ]:
# A RAG question.
print(run_agent("According to the docs, how are passwords currently handled?"))

In [ ]:
# A code-analysis question.
print(run_agent("How many functions and TODOs are in auth.py?"))

## 5. MCP Local (Opcional)

MCP es un transporte, no un servicio en la nube: un servidor MCP puede ejecutarse como un proceso local sobre `stdio`. La celda siguiente muestra cómo conectarse a un servidor MCP local si tienes uno configurado (por ejemplo, un servidor de sistema de archivos). Se omite de forma controlada cuando `LOCAL_MCP_COMMAND` no está establecido, para que el cuaderno siga ejecutándose de principio a fin sin él.

Nota de seguridad: un servidor MCP local se ejecuta con los permisos de tu usuario. Limítalo a un directorio de proyecto y valida sus salidas antes de actuar sobre ellas.


In [ ]:
import os

LOCAL_MCP_COMMAND = os.getenv("LOCAL_MCP_COMMAND")  # e.g. "npx -y @modelcontextprotocol/server-filesystem ./sample_project"

if not LOCAL_MCP_COMMAND:
    print("No LOCAL_MCP_COMMAND set — skipping the MCP demo. "
          "Set it to a local MCP server command to try this section.")
else:
    import asyncio
    from mcp import ClientSession, StdioServerParameters
    from mcp.client.stdio import stdio_client

    async def list_mcp_tools(command: str):
        parts = command.split()
        params = StdioServerParameters(command=parts[0], args=parts[1:])
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await session.list_tools()
                return [t.name for t in tools.tools]

    names = await list_mcp_tools(LOCAL_MCP_COMMAND)
    print("Local MCP server exposes tools:", names)

## Resumen

Construiste un asistente de ingeniería que funciona completamente en tu máquina:

- **Foundry Local** sirvió un modelo **Qwen** detrás de un endpoint compatible con OpenAI — de modo que el código del agente coincide con las lecciones en la nube.
- **Herramientas en Sandbox** brindaron al agente acceso a archivos y análisis de código sin salir del directorio del proyecto.
- **Chroma** proporcionó **RAG local** sobre la documentación.
- **MCP Local** mostró cómo reutilizar el ecosistema MCP sin conexión.

No se usó inferencia en la nube en ningún momento.

### Desafío

Amplía el agente local para:

1. **Trabajar con múltiples servidores MCP** — conectar un servidor de sistema de archivos y un servidor git y dejar que el agente elija entre ellos.
2. **Usar memoria local** — persistir un historial corto de conversaciones en el disco para que el asistente recuerde interacciones anteriores tras reiniciar el cuaderno.
3. **Soportar orquestación multiagente local** — añadir un segundo agente local (por ejemplo, un revisor) y hacer que ambos colaboren en una tarea.

En la próxima lección aprenderás cómo asegurar agentes de IA desplegados.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Descargo de responsabilidad**:
Este documento ha sido traducido utilizando el servicio de traducción automática [Co-op Translator](https://github.com/Azure/co-op-translator). Aunque nos esforzamos por la precisión, tenga en cuenta que las traducciones automatizadas pueden contener errores o inexactitudes. El documento original en su idioma nativo debe considerarse la fuente autorizada. Para información crítica, se recomienda una traducción profesional humana. No somos responsables de cualquier malentendido o interpretación errónea que surja del uso de esta traducción.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
